# Bölüm 7: Sohbet Uygulamaları Geliştirme
## OpenAI API Hızlı Başlangıç

Bu not defteri, [Azure OpenAI Örnekleri Deposu](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) temel alınarak uyarlanmıştır ve [Azure OpenAI](notebook-azure-openai.ipynb) hizmetlerine erişen not defterlerini içerir.

Python OpenAI API, birkaç değişiklikle Azure OpenAI Modelleri ile de çalışır. Farklılıklar hakkında daha fazla bilgi edinin: [Python ile OpenAI ve Azure OpenAI uç noktaları arasında nasıl geçiş yapılır](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Genel Bakış  
"Büyük dil modelleri, metni metne eşleyen fonksiyonlardır. Verilen bir giriş metin dizisi için büyük dil modeli, sonraki gelecek metni tahmin etmeye çalışır"(1). Bu "hızlı başlangıç" defteri, kullanıcılara üst düzey LLM kavramlarını, AML ile başlamaya yönelik temel paket gereksinimlerini, prompt tasarımına yumuşak bir giriş ve çeşitli kullanım durumlarından birkaç kısa örnek sunacaktır. 


## İçindekiler  

[Genel Bakış](#overview)  
[OpenAI Servisini Nasıl Kullanılır](#how-to-use-openai-service)  
[1. OpenAI Servisinizi Oluşturma](#1.-creating-your-openai-service)  
[2. Kurulum](#2.-installation)    
[3. Kimlik Bilgileri](#3.-credentials)  

[Kullanım Durumları](#use-cases)    
[1. Metni Özetle](#1.-summarize-text)  
[2. Metni Sınıflandır](#2.-classify-text)  
[3. Yeni Ürün İsimleri Üret](#3.-generate-new-product-names)  
[4. Bir Sınıflandırıcıyı İnce Ayar Yap](#4.fine-tune-a-classifier)  

[Kaynaklar](#references)


### İlk isteminizi oluşturun  
Bu kısa egzersiz, basit bir görev için OpenAI modeline istem göndermeye temel bir giriş sunacaktır: "özetleme".


**Adımlar**:  
1. Python ortamınıza OpenAI kütüphanesini yükleyin  
2. Standart yardımcı kütüphaneleri yükleyin ve oluşturduğunuz OpenAI Servisi için tipik OpenAI güvenlik kimlik bilgilerinizi ayarlayın  
3. Göreviniz için bir model seçin  
4. Model için basit bir istem oluşturun  
5. İsteğinizi model API'sine gönderin!


### 1. OpenAI'yi Yükleyin


In [ ]:
%pip install openai python-dotenv

### 2. Yardımcı kütüphaneleri içe aktarın ve kimlik bilgilerini oluşturun


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Doğru modeli bulmak  
GPT-4o ve GPT-4o mini gibi modeller doğal dili anlayabilir ve üretebilir, ayrıca farklı görevler için uygun çeşitli güç ve hız seviyeleriyle OpenAI platformunda mevcuttur.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. İstem Tasarımı  

"Büyük dil modellerinin sihri, bu tahmin hatasını çok miktarda metin üzerinde minimize etmek üzere eğitilmeleri sayesinde, modellerin bu tahminler için faydalı kavramları öğrenmeleridir. Örneğin, şu kavramları öğrenirler"(1):

* nasıl heceleme yapılır
* dilbilgisinin nasıl çalıştığı
* nasıl eşanlamlı ifade yapılır
* nasıl sorular cevaplanır
* nasıl sohbet edilir
* birçok dilde nasıl yazılır
* nasıl kod yazılır
* vb.

#### Büyük bir dil modeli nasıl kontrol edilir  
"Büyük bir dil modeline verilen tüm girdiler arasında, en etkili olanı kesinlikle metin istemidir(1).

Büyük dil modelleri birkaç şekilde çıktı üretmesi için yönlendirilebilir:

Talimat: Modelle ne istediğinizi söyleyin
Tamamlama: İstediğiniz şeyin başlangıcını modelin tamamlamasını sağlayın
Gösterim: Modele ne istediğinizi gösterin, şu yollardan biriyle:
İstemde birkaç örnek
İnce ayar eğitim veri setinde yüzlerce veya binlerce örnek"



#### İstemler oluşturmak için üç temel kılavuz vardır:

**Göster ve anlat**. Ne istediğinizi talimatlar, örnekler ya da ikisinin kombinasyonu yoluyla açıkça belirtin. Modelin bir öğeler listesini alfabetik sıraya göre sıralamasını veya bir paragrafı duyguya göre sınıflandırmasını istiyorsanız, isteğinizin bu olduğunu gösterin.

**Kaliteli veri sağlayın**. Bir sınıflandırıcı oluşturuyorsanız veya modelin bir deseni takip etmesini istiyorsanız, yeterli sayıda örnek olduğundan emin olun. Örneklerinizi mutlaka gözden geçirin — model genellikle temel yazım hatalarını fark edecek kadar zekidir ve size bir yanıt verir, ancak bunun kasıtlı olduğunu varsayabilir ve bu da yanıtı etkileyebilir.

**Ayarlarınızı kontrol edin.** Temperature ve top_p ayarları modelin yanıt üretme derecesini belirler. Eğer tek doğru yanıtın olduğu bir soru soruyorsanız, bunları düşük ayarlamak istersiniz. Daha çeşitli yanıtlar arıyorsanız, bunları daha yüksek ayarlamak isteyebilirsiniz. İnsanların bu ayarlarla yaptığı en büyük hata, bunların "zeka" veya "yaratıcılık" ayarları olduğunu varsaymaktır.


Kaynak: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Gönder!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Aynı çağrıyı tekrarlayın, sonuçlar nasıl karşılaştırılır?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Metni Özetle  
#### Zorluk  
Bir metin pasajının sonuna 'tl;dr:' ekleyerek metni özetleyin. Modelin ek bir talimat olmadan çeşitli görevleri nasıl yerine getirdiğini fark edin. Model davranışını değiştirmek ve aldığınız özeti kişiselleştirmek için tl;dr'den daha açıklayıcı istemlerle denemeler yapabilirsiniz(3).  

Son çalışmalar, büyük bir metin korpusunda ön eğitim yapıldıktan sonra belirli bir görev üzerinde ince ayar yapılarak birçok NLP görevi ve kıyaslama testi üzerinde önemli ilerlemeler sağlandığını göstermiştir. Genellikle mimarisi görevden bağımsız olsa da, bu yöntem yine de binlerce ya da on binlerce örnekten oluşan görev-özel ince ayar veri setleri gerektirir. Buna karşılık, insanlar genellikle sadece birkaç örnekten veya basit talimatlardan yeni bir dil görevini yerine getirebilirler - ki mevcut NLP sistemleri bunu çoğunlukla yapmada hala zorlanmaktadır. Burada, dil modellerinin ölçeklendirilmesinin görevden bağımsız, az örnekli performansı büyük ölçüde iyileştirdiğini ve bazen önceden en iyi ince ayar yöntemleri ile rekabet edebildiğini gösteriyoruz.  



Özetle  


# Birkaç kullanım durumu için alıştırmalar  
1. Metni Özetle  
2. Metni Sınıflandır  
3. Yeni Ürün İsimleri Üret


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Metni Sınıflandır  
#### Görev  
Öğrenme zamanında sağlanan kategorilere göre öğeleri sınıflandır. Aşağıdaki örnekte, hem kategoriler hem de sınıflandırılacak metin talebin içinde verilmektedir(*playground_reference).  

Müşteri Talebi: Merhaba, dizüstü bilgisayar klavyemin bir tuşu yakın zamanda kırıldı ve bir yedeğe ihtiyacım olacak:  

Sınıflandırılmış kategori:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Yeni Ürün İsimleri Oluşturma
#### Zorluk
Örnek kelimelerden ürün isimleri oluşturun. Burada, isimlerini oluşturacağımız ürün hakkında bilgiyi istemimize dahil ediyoruz. Ayrıca almak istediğimiz deseni göstermek için benzer bir örnek sağlıyoruz. Rastgeleliği artırmak ve daha yenilikçi cevaplar almak için sıcaklık değerini yüksek ayarladık.

Ürün açıklaması: Ev tipi milkshake yapıcı
Anahtar kelimeler: hızlı, sağlıklı, kompakt.
Ürün isimleri: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Ürün açıklaması: Her ayak numarasına uyabilen bir çift ayakkabı.
Anahtar kelimeler: uyarlanabilir, uyumlu, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Kaynaklar  
- [Openai Yemek Kitabı](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portalı](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [GPT modellerinin metin sınıflandırması için ince ayar yapmanın en iyi uygulamaları](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Daha Fazla Yardım İçin  
[OpenAI Ticarileştirme Ekibi](AzureOpenAITeam@microsoft.com) 


# Katkıda Bulunanlar
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
